# 1. 라이브러리 로드

In [ ]:
# unsloth 라이브러리 사전 설치 필요 

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# 2. 모델 로드

### 변수 설정

In [ ]:
model_name = "openai/gpt-oss-20b"
download_dir = "/workspace/model_repository"
output_dir = f"{download_dir}/gpt-oss-20b-lora-luckyvicky"
device = "cuda" if torch.cuda.is_available() else "cpu"
learning_rate = 2e-4

### LoRA 설정
lora_r = 4
lora_alpha = 8
lora_targets = ["q_proj", "v_proj"]  # ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
device = "cuda" if torch.cuda.is_available() else "cpu"


### 모델, 토크나이저 로드

In [ ]:
max_seq_length = 2048
dtype = None
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [ ]:
!nvidia-smi

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = lora_targets,
    lora_alpha = lora_alpha,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

 # 3. 데이터셋 로드
 ### 데이터셋 명: [Junnos/luckyvicky](https://huggingface.co/datasets/Junnos/luckyvicky)

In [ ]:
dataset = load_dataset("Junnos/luckyvicky", split="train")
print(dataset)

### train/validation 데이터셋 구분

### 템플릿 변환
#### gpt-oss-20b 모델의 chat-template.jinja 파일에 맞게 입력/출력 포맷팅

In [ ]:
def formatting_prompts_func(row):
    texts = []
    for input, out in zip(row["input"], row["output"]):
        messages = [
            {"role": "user", "content": input},
            {"role": "assistant", "content": out},
        ]

        # gpt-oss-20b 모델의 chat-template.jinja 에 기반하여 포맷팅
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

    return {"text": texts}

In [ ]:
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
)
print(dataset[:5])

In [ ]:
split_dataset = dataset.train_test_split(
    test_size=1/3,
    seed=42,
    shuffle=True,
)

train_dataset = split_dataset["train"]
val_dataset   = split_dataset["test"]

print(f"Total samples: {len(dataset)}")
print(f"Train samples: {len(train_dataset)}")
print(f"Valid samples: {len(val_dataset)}")

In [ ]:
### Trainer 설정
per_device_train_batch_size = 4    # 한번의 배치에서 몇 개의 데이터(row)를 학습할 것인지
num_train_epochs = 3               # 전체 데이터 학습을 몇 회 반복할 것인지
gradient_accumulation_steps = 4    # 몇 번의 학습(per_device_train_batch_size) 발생 후 가중치 업데이트를 할 것인지

# 4. Trainer 설정

In [ ]:
sfp_trainer = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,     # 4 * 4 = 16 이 하나의 스텝이 됨
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_strategy="steps",
        logging_steps = 1,
        eval_strategy="steps",
        eval_steps=1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = output_dir,
        report_to = "none", # Use TrackIO/WandB etc
    )

In [ ]:
trainer = SFTTrainer(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    model = model,
    tokenizer = tokenizer,
    args = sfp_trainer
)

In [ ]:
tokenizer.decode(trainer.train_dataset[50]["input_ids"])

# 학습 시작

In [ ]:
trainer.train()

# 학습 결과 확인

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps = []
train_losses = []

eval_steps = []
eval_losses = []

for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])

# 그래프 출력
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, label="Train Loss")
plt.plot(eval_steps, eval_losses, label="Validation Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# 결과물 저장

### 최종 생성된 Lora 어답터 가중치 파일과 토크나이저 파일

In [ ]:
print(model)

In [ ]:
messages = [
    {"role": "user", "content": "아 오늘 우리집에서 바퀴벌레가 나왔어. 완전 짜증나네 어떡하지?"}
]

kor_encoded_input = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")

# inputs = tokenizer(text, return_tensors='pt').to('cuda')
output = model.generate(**kor_encoded_input, max_new_tokens = 400)

print(output)

In [ ]:
decoded_output = tokenizer.decode(output[0], skip_special_tokens=False)
print(decoded_output)

In [ ]:
model.save_pretrained("model_repository/gpt-oss-20b-chat-lora-sft/lora")

In [ ]:
!nvidia-smi